# E17 — `e17_a`'s generation without B5 (raw + D1 + D1′) — run this one **only if the laptop says B5 was not adopted**

Registration: `research/e17/PREREG_E17.md` §3. Run **after the laptop has verified
`e17_a`'s training**. 50 files at E12's seeds.

**Attach two inputs:** **`uie-e17-code`**, and the **E17_train** notebook's **output**.
**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 2.5 h.
When done, download `/kaggle/working/e17_gen`.

In [ ]:
# --- 1. locate the code bundle (and the inputs) --------------------------------------
import hashlib, json, math
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E17_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-e17-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)


def output_of(arm):
    hits = sorted(p for p in root.rglob("PROVENANCE.json") if (p.parent / arm).is_dir()) if root.exists() else []
    return hits[0].parent if hits else None

SRC = output_of("e17_a")
if SRC is None:
    raise SystemExit("Attach the E17_train notebook's output too, then rerun.")
tprov = json.loads((SRC / "PROVENANCE.json").read_text())
CKPT = SRC / "e17_a" / "final.pt"
assert CKPT.is_file() and sha256(CKPT) == tprov["arms"]["e17_a"]["final_sha256"], \
    "e17_a: final.pt is not the one E17_train recorded"
print("checkpoint verified:", CKPT)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "e17", CODE / "research" / "e16", CODE / "research" / "e15",
          CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e15_common",
                      "e16_common", "e17_common")]:
    del sys.modules[name]
import uie
import e12_common as K
import e15_common as E15
import e16_common as E16
import e17_common as E17
manifest = json.loads((CODE / "E17_CODE_MANIFEST.json").read_text())
for mod in (uie, K, E15, E16, E17):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / E17.STREAM_FILE) != manifest["stream"]["sha256"]:
    bad.append(E17.STREAM_FILE)
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables and the pooled stream ({manifest['stream']['rows']:,} rows) verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
GPU = torch.cuda.get_device_name(0)
print("torch:", torch.__version__, "| gpu:", GPU)

In [ ]:
OUT_NAME, ARM_NAME, MODEL_NAME, B5_DECODE = 'e17_gen', E17.TRAIN_ARM, 'e17_a', False
PURPOSE = "E17 (research/e17/PREREG_E17.md): e17_a, decode d1p; generation only"

# --- 3. tables, the registered jobs, the model ---------------------------------------
from uie.controlled import StateSuffixTables
from uie.suffixes import SuffixTable
from uie.train import load_checkpoint

shipped = SuffixTable.load(CODE / "suffix_table.json")
tables = StateSuffixTables.load(CODE / "state_suffix_tables.json")
presence_slash = json.loads((CODE / "presence_slash_table.json").read_text())["table"]
device = torch.device("cuda")
OUT = WORK / OUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
jobs = sorted(E17.jobs(ARM_NAME), key=lambda j: (0 if j["mode"] == "free" else 1, j["label"], j["state"] or "", j["g"]))
print(f"{len(jobs)} files, {sum(j['n'] for j in jobs):,} URLs; decode {E17.decode_kwargs(B5_DECODE)}")
CKPT_SHA = {MODEL_NAME: sha256(CKPT)}
model, tok, _ = load_checkpoint(CKPT, device=device)
model.to(device).eval()
assert tok.presence_plan and tok.root_slash and tok.slash_plan and tok.vocab_size == 110
assert model.cfg.d_model == E16.D_MODEL, model.cfg.d_model
print(f"loaded {MODEL_NAME}: width {model.cfg.d_model}, vocab {tok.vocab_size}, max_len {model.cfg.max_len}")

In [ ]:
# --- 4. generate -----------------------------------------------------------------------
# PROVENANCE.json is rewritten after EVERY file; a rerun keeps a file only with a
# matching record -- anything else is made again: same seed, same bytes.
import platform
import time

B5 = False
ARM = E17.TRAIN_ARM
PROV = OUT / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"files": {}}
prov.update({"purpose": PURPOSE, "checkpoint_file": "final.pt", "code_commit": manifest["commit"],
             "checkpoints": CKPT_SHA, "tables": manifest["tables"], "decode": E17.decode_kwargs(B5),
             "environment": {"torch": torch.__version__, "python": platform.python_version(),
                             "gpu": GPU, "tldextract": metadata.version("tldextract")}})
t_start = time.time()
done = 0
for j in jobs:
    path = OUT / j["file"]
    rec = prov["files"].get(j["file"])
    if rec and path.is_file() and sha256(path) == rec["sha256"]:
        done += 1
        continue
    t0 = time.time()
    trace, qtrace = [], []
    if j["mode"] == "free":
        urls = E17.free_urls(model, tok, j["label"], j["g"], shipped, presence_slash, b5=B5,
                             device=device, plan_trace=trace, query_trace=qtrace if B5 else None)
    else:
        urls = E17.control_urls(model, tok, j["label"], j["state"], j["g"], tables, presence_slash,
                                b5=B5, device=device, plan_trace=trace,
                                query_trace=qtrace if B5 else None)
    assert len(urls) == j["n"], (j["file"], len(urls))
    path.write_text(json.dumps(urls))
    prov["files"][j["file"]] = {**{k: v for k, v in j.items() if k != "file"},
                                "sha256": sha256(path), "seconds": round(time.time() - t0, 1),
                                "plan_trace_mean": sum(trace) / len(trace) if trace else None,
                                "query_moved_mean": sum(qtrace) / len(qtrace) if qtrace else None,
                                "query_rows_moved": sum(x > 0 for x in qtrace) if qtrace else None}
    PROV.write_text(json.dumps(prov, indent=1))
    done += 1
    el = (time.time() - t_start) / 60
    print(f"  {done:>3}/{len(jobs)}  {j['file']:<46} {time.time() - t0:6.0f}s  ({el:.0f} min so far)",
          flush=True)
print(f"\nall {len(jobs)} files written. Now download {OUT}.")